# English Listening Video Generator (方案 B — Grouped Multi-Character)

Generates a ~13min English listening practice video with:
- LLM script (SenseNova DeepSeek V4 Flash, 18 dialogue lines + IPA + 繁中)
- 3 frontier character/scene images (3D cartoon style)
- Grouped Seedance2 video clips (方案 B: merges consecutive lines, both char refs)
- Kokoro TTS (English) + edge-tts (Chinese) + loudnorm
- FFmpeg + Pillow composition with bilingual subtitles

**Colab GPU recommended** for Kokoro TTS speed.

## Setup
1. Run Cell 1 to install dependencies (~3min)
2. Run Cell 2 to clone the repo from GitHub (auto-downloads all scripts)
3. Set your API keys in Cell 3
4. Run Cell 4 to generate the video

## Cell 1: Install Dependencies

In [ ]:
# Install system dependencies
!apt-get update -qq && apt-get install -y -qq ffmpeg git

# Install Python dependencies
!pip install -q kokoro soundfile torch edge-tts Pillow opencc-python-reimplemented
!pip install -q cn2an pypinyin ordered_set jieba

# Install CJK fonts for Pillow rendering
!apt-get install -y -qq fonts-noto-cjk fonts-dejavu-core

# Create output directory
import os
os.makedirs('/content/output', exist_ok=True)

print('Dependencies installed.')
print(f'FFmpeg: {os.popen("ffmpeg -version 2>/dev/null | head -1").read().strip()}')

## Cell 2: Clone Repository from GitHub

Automatically downloads all scripts from the GitHub repo. No manual upload needed.

In [ ]:
import os

REPO_URL = 'https://github.com/collinsgraciano/colab_listening_b.git'
REPO_DIR = '/content/listening_b'

# Remove old clone if exists
if os.path.exists(REPO_DIR):
    !rm -rf {REPO_DIR}

!git clone -q {REPO_URL} {REPO_DIR}

# Verify all required scripts are present
required = ['mcp_client.py', 'llm_client.py', 'tts_engine.py', 'timeline.py',
            'grouping_b.py', 'video_compose.py', 'pipeline.py', 'topics.json',
            'topic_manager.py']
missing = [f for f in required if not os.path.exists(os.path.join(REPO_DIR, f))]
if missing:
    print(f'Missing files: {missing}')
else:
    print('All scripts cloned successfully!')

## Cell 3: Set API Keys

### MCP OAuth Token
Get your TJGenerators MCP OAuth token from your local machine:
```python
import json
tokens = json.load(open(r'C:\Users\Administrator\.codely-cli\mcp-oauth-tokens.json'))
token = next(t['token']['accessToken'] for t in tokens if t['serverName'] == 'TJGenerators')
print(token)
```

### SenseNova API Key
Get your SenseNova API key from the SenseNova console: https://console.senseNova.cn
```python
# The key looks like: sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
```

In [ ]:
import os

# Paste your MCP OAuth token here
MCP_TOKEN = 'PASTE_YOUR_MCP_TOKEN_HERE'

# Paste your SenseNova API key here
SENSENOVA_API_KEY = 'PASTE_YOUR_API_KEY_HERE'

if MCP_TOKEN == 'PASTE_YOUR_MCP_TOKEN_HERE':
    print('Please paste your MCP token above!')
elif SENSENOVA_API_KEY == 'PASTE_YOUR_API_KEY_HERE':
    print('Please paste your SenseNova API key above!')
else:
    os.environ['SENSENOVA_API_KEY'] = SENSENOVA_API_KEY
    print(f'MCP Token set ({len(MCP_TOKEN)} chars)')
    print(f'SenseNova API Key set ({len(SENSENOVA_API_KEY)} chars)')

## Cell 4: Generate Video

Set the topic and CEFR level, then run. The pipeline takes ~15-25 minutes.

In [ ]:
import sys, os
sys.path.insert(0, '/content/listening_b')

# TOPIC: leave empty ('') to auto-select a random unused topic from topics.json
#        (selected topic is recorded to used_topics.json to prevent duplicates)
TOPIC = ''  # e.g. 'Doing Laundry' or leave '' for random
CEFR = 'A2'                    # A1, A2, B1, B2, C1, C2
NUM_LINES = 18                 # Number of dialogue lines (default 18, can set 12-24)
OUTPUT_DIR = '/content/output'

# Build command: if TOPIC empty, omit --topic so pipeline picks random + marks used
cmd = f'cd /content/listening_b && python pipeline.py '
if TOPIC.strip():
    cmd += f'--topic "{TOPIC}" '
cmd += f'--cefr {CEFR} --num-lines {NUM_LINES} --output {OUTPUT_DIR} '
cmd += f'--mcp-token {MCP_TOKEN} --api-key {SENSENOVA_API_KEY}'

print('Running:')
print(cmd)
print()
!{cmd}

## Cell 5: Download Result

In [ ]:
from google.colab import files
import os

video_path = '/content/output/videos/final_video.mp4'
if os.path.exists(video_path):
    size_mb = os.path.getsize(video_path) / (1024*1024)
    print(f'Video: {video_path} ({size_mb:.1f}MB)')
    files.download(video_path)
else:
    print('Video not found. Check pipeline output above for errors.')

## Cell 6: Preview Video (optional)

In [ ]:
from IPython.display import HTML
from base64 import b64encode
import os

video_path = '/content/output/videos/final_video.mp4'
if os.path.exists(video_path):
    with open(video_path, 'rb') as f:
        video_b64 = b64encode(f.read()).decode()
    html = f'''
    <video width="640" height="360" controls>
        <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
    </video>
    '''
    display(HTML(html))
else:
    print('Video not found.')